## Máster en Big Data y Data Science

### Metodologías de gestión y diseño de proyectos de big data

#### AP2 - Modelado y evaluación

---

En esta libreta se realiza la experimentación para generación del modelo de predicción objetivo del proyecto y la evaluación del mismo.
La versión del dataset a utilizar es la obtenida a partir de las operaciones de transformación.

In [143]:
# Se importan las librerías necesarias y se suprimen las advertencias
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore',category=FutureWarning)
warnings.filterwarnings('ignore',category=UserWarning)

In [144]:
import mlflow
import mlflow.sklearn
from datetime import datetime

# Configuración de MLflow
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("Proyecto 13MBID-ABR2526 - Experimentación original")

<Experiment: artifact_location='file:d:/Codigo/13MBID_Porftolio_2526/notebooks/mlruns/876894971671164557', creation_time=1762973609354, experiment_id='876894971671164557', last_update_time=1762973609354, lifecycle_stage='active', name='Proyecto 13MBID-ABR2526 - Experimentación original', tags={}>

In [131]:
# Lectura de los datos
df = pd.read_csv('../data/processed/bank-additional-full_preprocessed.csv', sep=';')
df.head(5)

,age,job,marital,education,housing,loan,contact,month,day_of_week,duration,...,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,contacts_diff,contacted_before
0,56,housemaid,married,basic.4y,0,0,telephone,may,mon,261,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1,0
1,57,services,married,high.school,0,0,telephone,may,mon,149,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1,0
2,37,services,married,high.school,1,0,telephone,may,mon,226,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1,0
3,40,admin.,married,basic.6y,0,0,telephone,may,mon,151,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1,0
4,56,services,married,high.school,0,1,telephone,may,mon,307,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,1,0


In [132]:
# Se divide el dataset en variables predictoras y variable objetivo
X = df.drop('y', axis=1)
y = df['y']

In [133]:
# Se genera el conjunto de entrenamiento y prueba con estratificación
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [134]:
# Se separan las columnas numéricas
numerical_columns=X_train.select_dtypes(exclude='object').columns
display(numerical_columns)

categorical_columns=X_train.select_dtypes(include='object').columns
display(categorical_columns)

Index(['age', 'housing', 'loan', 'duration', 'campaign', 'previous',
       'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m',
       'nr_employed', 'contacts_diff', 'contacted_before'],
      dtype='object')

Index(['job', 'marital', 'education', 'contact', 'month', 'day_of_week',
       'poutcome'],
      dtype='object')

In [135]:
# Se verifica la distribución de la variable objetivo en el conjunto de entrenamiento
y_train.value_counts()

y
0    27179
1     3406
Name: count, dtype: int64

In [136]:
# Se crea un pipeline para preprocesamiento de datos
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler  

# Pipeline para valores numéricos
num_pipeline = Pipeline(steps=[
    ('RobustScaler', RobustScaler())
])

# Pipeline para valores categóricos
cat_pipeline = Pipeline(steps=[
    ('OneHotEncoder', OneHotEncoder(drop='first',sparse_output=False))
])

# Se configuran los preprocesadores
preprocessor_full = ColumnTransformer([
    ('num_pipeline', num_pipeline, numerical_columns),
    ('cat_pipeline', cat_pipeline, categorical_columns)
]).set_output(transform='pandas')

In [137]:
preprocessor_train_valid = ColumnTransformer([
    ('num_pipeline', num_pipeline, numerical_columns),
    ('cat_pipeline', cat_pipeline, categorical_columns)
]).set_output(transform='pandas')

In [139]:
# Se ajusta y transforma el conjunto de entrenamiento y prueba
x_train_prep = preprocessor_full.fit_transform(X_train)
x_test_prep = preprocessor_full.transform(X_test)

In [140]:
# Se aplica submuestreo a los datos preprocesados
from sklearn.utils import resample

# Combinar los datos preprocesados con las etiquetas
train_data = x_train_prep.copy()
train_data['target'] = y_train.reset_index(drop=True)

# Separar por clase
class_0 = train_data[train_data['target'] == 0]
class_1 = train_data[train_data['target'] == 1]

# Encontrar la clase minoritaria
min_count = min(len(class_0), len(class_1))

# Submuestreo balanceado - tomar una muestra igual al tamaño de la clase minoritaria
class_0_balanced = resample(class_0, n_samples=min_count, random_state=42)
class_1_balanced = resample(class_1, n_samples=min_count, random_state=42)

# Combinar las clases balanceadas
balanced_data = pd.concat([class_0_balanced, class_1_balanced])

# Separar características y objetivo
x_train_resampled = balanced_data.drop('target', axis=1)
y_train_resampled = balanced_data['target']

print(f"Tamaño original: {len(x_train_prep)}")
print(f"Tamaño balanceado: {len(x_train_resampled)}")
print(f"Distribución balanceada: {y_train_resampled.value_counts()}")

Tamaño original: 30585
Tamaño balanceado: 5438
Distribución balanceada: target
0.0    2719
1.0    2719
Name: count, dtype: int64


In [171]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from mlflow.models import infer_signature

# Cross validation modificada para registrar en MLflow
def cross_val_mlflow(model, model_name, params = None):
    with mlflow.start_run(run_name=f"{model_name}"):
        # Validación cruzada  por cada métrica y cálculo de medias y desviaciones estándar
        f1_scores = cross_val_score(model, x_train_resampled, y_train_resampled, cv=5, scoring='f1')
        f1_mean = f1_scores.mean()
        f1_std = f1_scores.std()

        recall_scores = cross_val_score(model, x_train_resampled, y_train_resampled, cv=5, scoring='recall')
        recall_mean = recall_scores.mean()
        recall_std = recall_scores.std()

        precision_scores = cross_val_score(model, x_train_resampled, y_train_resampled, cv=5, scoring='precision')
        precision_mean = precision_scores.mean()
        precision_std = precision_scores.std()

        accuracy_scores = cross_val_score(model, x_train_resampled, y_train_resampled, cv=5, scoring='accuracy')
        accuracy_mean = accuracy_scores.mean()
        accuracy_std = accuracy_scores.std()

        # Se entrena el modelo
        model.fit(x_train_resampled, y_train_resampled)
        y_pred = model.predict(x_test_prep)

        # Registro del modelo en MLflow
        signature = infer_signature(x_train_resampled, y_pred)
        
        test_f1 = f1_score(y_test, y_pred)
        test_recall = recall_score(y_test, y_pred)
        test_precision = precision_score(y_test, y_pred)
        test_accuracy = accuracy_score(y_test, y_pred)

        # Registro de parámetros en MLflow
        if params:
            mlflow.log_params(params)
        else:
            mlflow.log_params(model.get_params())
        
        mlflow.log_params({
            "train_samples": len(x_train_resampled),
            "test_samples": len(x_test_prep),
            "balancing_method": "undersampling",
            "cv_folds": 5
        })

        # Registro de métricas
        mlflow.log_metrics({
            "cv_f1_mean": f1_mean,
            "cv_recall_mean": recall_mean,
            "cv_precision_mean": precision_mean,
            "cv_accuracy_mean": accuracy_mean,
            "cv_f1_std": f1_std,
            "cv_recall_std": recall_std,
            "cv_precision_std": precision_std,
            "cv_accuracy_std": accuracy_std,
        })
        mlflow.log_metrics({
            "test_f1": test_f1,
            "test_recall": test_recall,
            "test_precision": test_precision,
            "test_accuracy": test_accuracy
        })

        # Registro del modelo
        mlflow.sklearn.log_model(
            model, 
            artifact_path="model", 
            signature=signature
        )
        
        print(f"Modelo {model_name} registrado en MLflow con ID de ejecución: {mlflow.active_run().info.run_id}")

        return model, {
            "f1_mean": f1_mean,
            "recall_mean": recall_mean,
            "precision_mean": precision_mean,
            "accuracy_mean": accuracy_mean,
            "test_f1": test_f1,
            "test_recall": test_recall,
            "test_precision": test_precision,
            "test_accuracy": test_accuracy
        }

In [172]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Resultados
resultados = {}

# Metodo 1 - regresión
lr = LogisticRegression(C=1,penalty='l2',solver='liblinear',random_state=1,max_iter=100,tol=0.000000001)
model_lr, metrics_lr = cross_val_mlflow(lr, "Logistic Regression")
resultados['Logistic Regression'] = metrics_lr

# Metodo 2 - SVC
svc = LinearSVC(max_iter=10000,tol=0.001)
model_svc, metrics_svc = cross_val_mlflow(svc, "Linear SVC")
resultados['Linear SVC'] = metrics_svc

# Metodo 3 - KNN
knc = KNeighborsClassifier(n_neighbors=7)
model_knc, metrics_knc = cross_val_mlflow(knc, "K-Nearest Neighbors")
resultados['K-Nearest Neighbors'] = metrics_knc

# Metodo 4 - Decision Tree
tree = DecisionTreeClassifier()
model_tree, metrics_tree = cross_val_mlflow(tree, "Decision Tree Classifier")
resultados['Decision Tree Classifier'] = metrics_tree

Modelo Logistic Regression registrado en MLflow con ID de ejecución: d76a68148b8748dd8063f066f947edd2
Modelo Linear SVC registrado en MLflow con ID de ejecución: d48d3e58a8ea443e982d5f5ee8c6cd5a
Modelo K-Nearest Neighbors registrado en MLflow con ID de ejecución: 69a9b96377b4433f85c1a184e3f1ec5e
Modelo Decision Tree Classifier registrado en MLflow con ID de ejecución: 84d79fb6bcc4462aa77302135f096792


Comparación de modelos para seleccionar posteriormente

In [173]:
df_comparacion = pd.DataFrame(resultados).T
df_comparacion = df_comparacion.round(5)
df_comparacion = df_comparacion.sort_values(by='test_f1', ascending=False)

print(df_comparacion)
print("El mejor modelo según F1 en test es:", df_comparacion.index[0])
print("Valor de F1 en test:", df_comparacion.iloc[0]['test_f1'])
print("Valor de Recall en test:", df_comparacion.iloc[0]['test_recall'])
print("Valor de Precision en test:", df_comparacion.iloc[0]['test_precision'])
print("Valor de Accuracy en test:", df_comparacion.iloc[0]['test_accuracy'])

                          f1_mean  recall_mean  precision_mean  accuracy_mean  \
K-Nearest Neighbors       0.57909      0.59582         0.56356        0.56712   
Decision Tree Classifier  0.69602      0.74256         0.66385        0.67856   
Logistic Regression       0.52402      0.51968         0.52895        0.52758   
Linear SVC                0.52454      0.52042         0.52921        0.52795   

                          test_f1  test_recall  test_precision  test_accuracy  
K-Nearest Neighbors       0.18985      0.50529         0.11688        0.52007  
Decision Tree Classifier  0.18256      0.41833         0.11676        0.58310  
Logistic Regression       0.11989      0.29142         0.07547        0.52387  
Linear SVC                0.11762      0.28320         0.07422        0.52713  
El mejor modelo según F1 en test es: K-Nearest Neighbors
Valor de F1 en test: 0.18985
Valor de Recall en test: 0.50529
Valor de Precision en test: 0.11688
Valor de Accuracy en test: 0.52007


---

#### Predicción con datos nuevos (sin clasificar)

In [97]:
df_nuevos = pd.read_csv('../data/raw/bank-additional-new.csv')
df_nuevos.head(15)

,age,job,marital,education,housing,loan,contact,month,day_of_week,duration,campaign,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,contacted_before
0,56,housemaid,married,basic.4y,no,no,telephone,may,mon,261,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,NaN
1,57,services,married,high.school,no,no,telephone,may,mon,149,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,NaN
2,37,services,married,high.school,yes,no,telephone,may,mon,226,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,NaN
3,40,admin.,married,basic.6y,no,no,telephone,may,mon,151,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,NaN
4,56,services,married,high.school,no,yes,telephone,may,mon,307,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,NaN
5,45,services,married,basic.9y,no,no,telephone,may,mon,198,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,NaN
6,59,admin.,married,professional.course,no,no,telephone,may,mon,139,1,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,NaN
7,29,admin.,single,university.degree,no,no,cellular,apr,thu,298,4,0,nonexistent,-1.8,93.075,-47.1,1.410,5099.1,no,NaN
8,33,technician,married,professional.course,no,yes,cellular,apr,thu,667,1,0,nonexistent,-1.8,93.075,-47.1,1.410,5099.1,no,NaN


In [98]:
# Diagnosticar el problema con los nuevos datos
print("Información del conjunto de datos nuevos:")
print(f"Forma: {df_nuevos.shape}")
print("\nTipos de datos:")
print(df_nuevos.dtypes)
print("\nValores nulos:")
print(df_nuevos.isnull().sum())
print("\nColumnas categóricas en nuevos datos:")
print(df_nuevos.select_dtypes(include='object').columns.tolist())
print("\nColumnas numéricas en nuevos datos:")
print(df_nuevos.select_dtypes(exclude='object').columns.tolist())

Información del conjunto de datos nuevos:
Forma: (9, 20)

Tipos de datos:
age                   int64
job                  object
marital              object
education            object
housing              object
loan                 object
contact              object
month                object
day_of_week          object
duration              int64
campaign              int64
previous              int64
poutcome             object
emp_var_rate        float64
cons_price_idx      float64
cons_conf_idx       float64
euribor3m           float64
nr_employed         float64
y                    object
contacted_before    float64
dtype: object

Valores nulos:
age                 0
job                 0
marital             0
education           0
housing             0
loan                0
contact             0
month               0
day_of_week         0
duration            0
campaign            0
previous            0
poutcome            0
emp_var_rate        0
cons_price_idx      0
cons_c

In [99]:
# Comparar con los datos de entrenamiento originales
print("Comparación de columnas:")
print(f"Columnas en datos originales: {list(X.columns)}")
print(f"Columnas en datos nuevos: {list(df_nuevos.columns)}")

print("\nColumnas que están en nuevos pero no en originales:")
new_cols = set(df_nuevos.columns) - set(X.columns)
print(new_cols)

print("\nColumnas que están en originales pero no en nuevos:")
missing_cols = set(X.columns) - set(df_nuevos.columns)
print(missing_cols)

Comparación de columnas:
Columnas en datos originales: ['age', 'job', 'marital', 'education', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'previous', 'poutcome', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'contacts_diff', 'contacted_before']
Columnas en datos nuevos: ['age', 'job', 'marital', 'education', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'previous', 'poutcome', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'y', 'contacted_before']

Columnas que están en nuevos pero no en originales:
{'y'}

Columnas que están en originales pero no en nuevos:
{'contacts_diff'}


In [115]:
import numpy as np
# Se hace la predicción con los nuevos datos
# Primero, eliminar la columna objetivo si existe y preparar las características
X_new = df_nuevos.drop('y', axis=1) if 'y' in df_nuevos.columns else df_nuevos.copy()
X_new["contacts_diff"] = X_new["campaign"] - X_new["previous"]
# Conversión de columnas binarias 'housing' y 'loan' a 0/1
binary_columns = ['housing', 'loan']
for col in binary_columns:
    X_new[col] = X_new[col].map({'yes': 1, 'no': 0})

# Asegurar que las columnas estén en el mismo orden que en el entrenamiento
X_new = X_new[X.columns]

# Manejar la columna contacted_before para que coincida con el formato de entrenamiento
# En entrenamiento: 0 (no), 1 (yes) (int)
# Si "previous" > 0, entonces "contacted_before" = 1, sino 0.
# De esta manera también correjimos incoherencias entre previous y contacted_before
X_new['contacted_before'] = np.where(X_new['previous'] > 0, 1, 0)

# Asegurar que contacted_before sea de tipo int como en entrenamiento
X_new['contacted_before'] = X_new['contacted_before'].astype('int')

# Transformar los nuevos datos usando el mismo preprocesador y predecir
try:
    x_new_prep = preprocessor_full.transform(X_new)
    
    y_new_pred = tree.predict(x_new_prep)
    print(f"\nPredicciones: {y_new_pred}")
    
    predictions_df = pd.DataFrame({
        'Cliente': range(1, len(y_new_pred) + 1),
        'Predicción_Numérica': y_new_pred,
        'Suscribirá': ['No' if pred == 0 else 'Sí' for pred in y_new_pred]
    })
    print("\nResultados detallados:")
    print(predictions_df.to_string(index=False))
    
    # Resumen de predicciones
    pred_counts = pd.Series(y_new_pred).value_counts()
    print("\nResumen de predicciones:")
    for pred_val, count in pred_counts.items():
        label = 'No realizará un depósito' if pred_val == 0 else 'Sí realizará un depósito'
        print(f"  {label}: {count} clientes ({count/len(y_new_pred)*100:.1f}%)")
    
except Exception as e:
    print(f"Error durante el preprocesamiento o predicción: {e}")
    print("Información adicional para depuración:")
    print(f"Tipos de datos en X_new:\n{X_new.dtypes}")


Predicciones: [0. 1. 1. 0. 1. 0. 0. 1. 0.]

Resultados detallados:
 Cliente  Predicción_Numérica Suscribirá
       1                  0.0         No
       2                  1.0         Sí
       3                  1.0         Sí
       4                  0.0         No
       5                  1.0         Sí
       6                  0.0         No
       7                  0.0         No
       8                  1.0         Sí
       9                  0.0         No

Resumen de predicciones:
  No realizará un depósito: 5 clientes (55.6%)
  Sí realizará un depósito: 4 clientes (44.4%)
